In [ ]:
import copy
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

In [ ]:
data = pd.read_csv(r"C:\Users\ykila\Desktop\iCloud\iCloudDrive\Projects\Active\Code\Python\LLM\Aalto\Dataframe.csv")
data["log_pSat"] = np.log10(data["pSat_Pa"])  # compress 10+ orders of magnitude into a tractable range

In [ ]:
# ECFP4 (Morgan radius=2) fingerprints generated from SMILES.
# Same featurisation as v1, v3, v4 — the only feature set across all four versions.
def smiles_to_ecfp(smi, n_bits=2048, radius=2):
    arr = np.zeros(n_bits, dtype=np.uint8)
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fps = np.stack([smiles_to_ecfp(s) for s in data["SMILES"]])
print(f"generated {fps.shape[1]}-bit ECFP4 for {fps.shape[0]} molecules")

In [ ]:
X = fps  # ECFP4 fingerprints are the only feature set across all four versions
y = data["log_pSat"].values
print(f"X shape: {X.shape}  (pure {X.shape[1]}-bit ECFP4)")

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )


In [ ]:
# NEW vs v1: carve 10% off training as a validation set so early stopping
# and best-state selection have something to score against without touching X_test.
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [ ]:
train_dataset = TensorDataset(
    torch.tensor(X_tr, dtype=torch.float32),
    torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1),
)
val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32).unsqueeze(1),
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).unsqueeze(1),
)

In [ ]:
train_dl = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dl = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [ ]:
model = nn.Sequential(
    nn.Linear(X_tr.shape[1], 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 1),
)

device = "mps" if torch.backends.mps.is_available() else "cpu"

model = model.to(device)

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss()  # Huber: more robust to outlier residuals than MSE

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)

In [ ]:
best_val_mae = float("inf")
best_state = None
patience = 15
since_best = 0

for epoch in range(200):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()

    # Validation pass — drives both the LR scheduler and early stopping
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            preds.append(model(x))
            trues.append(y)
    val_mae = mean_absolute_error(torch.cat(trues).cpu().numpy(),
                                  torch.cat(preds).cpu().numpy())
    print(f"epoch {epoch} val MAE = {val_mae:.4f}")

    scheduler.step(val_mae)

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state = copy.deepcopy(model.state_dict())
        since_best = 0
    else:
        since_best += 1
        if since_best >= patience:
            print(f"early stop at epoch {epoch}, best val MAE = {best_val_mae:.4f}")
            break

# Restore best weights and report final test MAE — first time we touch X_test
model.load_state_dict(best_state)
model.eval()
test_preds = []
with torch.no_grad():
    for x, _ in test_dl:
        test_preds.append(model(x.to(device)).cpu().numpy())
test_preds = np.vstack(test_preds).flatten()
print(f"\nfinal test MAE = {mean_absolute_error(y_test, test_preds):.4f}")